In [1]:
!pip install Groq

In [2]:
!pip install langchain-groq langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.1/591.1 kB 14.0 MB/s  0:00:00
  Attempting uninstall: langchain-protocol
    Found existing installation: langchain-protocol 0.0.20
    Uninstalling langchain-protocol-0.0.20:
      Successfully uninstalled langchain-protocol-0.0.20
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.7.4
    Uninstalling langchain-core-1.7.4:
      Successfully uninstalled langchain-core-1.7.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [langchain-groq]


In [5]:
import os
from getpass import getpass

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

# Get API Key
os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

# Initialize LLM
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

# Send messages
response = llm.invoke([
    SystemMessage(content="Act as a professional"),
    HumanMessage(content="tell me my name Gokul")
])

# Display output
print(response.content)

Enter your Groq API Key: ··········
Your name is Gokul.


In [7]:
import os
import json
from getpass import getpass

from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

# --------------------------------
# 1. API KEY
# --------------------------------

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API Key: ")

# --------------------------------
# 2. INITIALIZE LLM
# --------------------------------

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

# --------------------------------
# 3. LONG-TERM MEMORY
# --------------------------------

memory_file = "user_memory.json"

if os.path.exists(memory_file):
    with open(memory_file, "r") as file:
        user_memory = json.load(file)
else:
    user_memory = {}

# --------------------------------
# 4. CONVERSATION HISTORY
# --------------------------------

chat_history = []

# --------------------------------
# 5. CHAT FUNCTION
# --------------------------------

def chat(user_input):

    global user_memory

    system_prompt = f"""
    You are a professional AI assistant.

    User information stored in memory:
    {json.dumps(user_memory)}

    Instructions:
    1. Remember information shared by the user.
    2. If the user tells you their name, identify it.
    3. Use the user's name when relevant.
    4. Answer naturally and professionally.
    5. Never invent personal information.
    """

    messages = [
        SystemMessage(content=system_prompt)
    ]

    messages.extend(chat_history)

    messages.append(
        HumanMessage(content=user_input)
    )

    response = llm.invoke(messages)

    # --------------------------------
    # 6. EXTRACT USER INFORMATION
    # --------------------------------

    extraction_prompt = f"""
    Analyze the following user message.

    Current memory:
    {json.dumps(user_memory)}

    User message:
    {user_input}

    Extract any personal information explicitly shared by the user.

    Return ONLY valid JSON.

    Example:
    {{"name": "Dharani"}}

    If no new personal information is shared, return:
    {{}}

    Do not guess or invent information.
    """

    extraction_response = llm.invoke(extraction_prompt)

    try:
        new_memory = json.loads(extraction_response.content)

        # Update memory
        user_memory.update(new_memory)

        # Save permanently
        with open(memory_file, "w") as file:
            json.dump(user_memory, file, indent=4)

    except json.JSONDecodeError:
        print("Memory extraction failed.")

    # --------------------------------
    # 7. SAVE CHAT HISTORY
    # --------------------------------

    chat_history.append(
        HumanMessage(content=user_input)
    )

    chat_history.append(
        AIMessage(content=response.content)
    )

    return response.content


# --------------------------------
# 8. INTERACTIVE CHATBOT
# --------------------------------

print("AI Assistant Started!")
print("Type 'exit' to stop.\n")

while True:

    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("AI: Goodbye!")
        break

    answer = chat(user_input)

    print("\nAI:", answer)
    print()

Enter your Groq API Key: ··········
AI Assistant Started!
Type 'exit' to stop.

You: what is my name

AI: Your name is **Jim Rat**.

You: exit
AI: Goodbye!
